# 05 · Evaluation frameworks

An evaluation has five parts: test cases, metric functions, a runner, aggregate
summaries, and thresholds that gate a release. This notebook builds each part without
external dependencies, so you can see what the frameworks do internally.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from evaluation import (keyword_recall, groundedness, retrieval_metrics,
                        doc_ids_from_chunks, run_eval, score_from_judge)
from rag import RAGPipeline
from embeddings import get_embedder
from llm_utils import get_llm
from sample_data import DOCS, QA_CASES

pipeline = RAGPipeline(get_llm(), get_embedder(), k=2)
pipeline.index(DOCS)
print("Cases:", len(QA_CASES))

## 1. Metrics

Each metric takes a case and the system's result. Returning `None` means the metric
does not apply to that case, and it is left out of the average.

In [ ]:
def answer_keywords(case, result):
    if case["expect_abstain"]:
        return None
    return keyword_recall(result["answer"] or "", case["keywords"])

def abstained(case, result):
    if not case["expect_abstain"]:
        return None
    return 1.0 if (result["answer"] or "").lower().startswith("i don't know") else 0.0

def retrieval_mrr(case, result):
    if case["relevant_doc"] is None:
        return None
    ranked = doc_ids_from_chunks(result["sources"])
    return retrieval_metrics(ranked, [case["relevant_doc"]], k=max(len(ranked), 1))["mrr"]

def grounded(case, result):
    if case["expect_abstain"]:
        return None
    return groundedness(result["answer"] or "", result["context"])

METRICS = {
    "answer_keywords": answer_keywords,
    "abstained": abstained,
    "retrieval_mrr": retrieval_mrr,
    "groundedness": grounded,
}

## 2. Run the suite

In [ ]:
def system_under_test(question):
    return pipeline.answer(question)

report = run_eval(QA_CASES, system_under_test, METRICS)
for row in report["rows"]:
    print(row)
print("\nSummary:", report["summary"])

## 3. Gate the release on thresholds

These thresholds are placeholders. Set them from a baseline measured on your own data,
and keep them in version control next to the test cases.

In [ ]:
THRESHOLDS = {"answer_keywords": 0.8, "abstained": 1.0, "retrieval_mrr": 0.5}

gate_failures = []
for name, minimum in THRESHOLDS.items():
    value = report["summary"].get(name)
    if value is None or value < minimum:
        gate_failures.append((name, value, minimum))

print("Gate:", "PASS" if not gate_failures else "FAIL")
for name, value, minimum in gate_failures:
    print(f"  {name}: {value} (required >= {minimum})")

## 4. LLM-as-judge

A judge model scores answers against a rubric. It scales better than human review, but
it must be validated against human labels first, because it can be inconsistent or biased.

The offline mock cannot judge, so this section shows the plumbing with a simulated reply.
Call `judge_groundedness` only when `LLM_PROVIDER=openai` is set.

In [ ]:
RUBRIC = ("Rate how well the answer is supported by the context on a 1-5 scale. "
          "5 = every claim is supported; 1 = unsupported. Reply with the number first.")

def judge_groundedness(case, result):
    if case["expect_abstain"]:
        return None
    messages = [
        {"role": "system", "content": RUBRIC},
        {"role": "user", "content": (f"Context:\n{result['context']}\n\n"
                                     f"Question: {case['question']}\nAnswer: {result['answer']}")},
    ]
    reply = get_llm().chat(messages).content or ""
    return score_from_judge(reply)

simulated_reply = "4 - Most claims are supported by the refund policy context."
print("Parsed judge score:", score_from_judge(simulated_reply))

## 5. Where this goes next

- **Regression suite:** every production bug becomes a permanent case, tagged with its origin.
- **Run on every change:** prompts, models, chunking, and indexes all need the same suite.
- **Judge calibration:** measure agreement between the judge and human reviewers, and re-check it when the judge model changes.
- **Frameworks:** Ragas (RAG metrics), DeepEval, promptfoo (prompt and model comparisons), and TruLens cover much of this with more metrics and dashboards. Use them once you understand the parts above, so you can check what they report.
- **Production monitoring:** apply the same rule-based metrics to sampled live traffic, and track thumbs-down rates and refusal rates over time (see the evaluation guide).